# Advanced AgentMind Tutorial: Building Production Systems

This tutorial covers advanced topics for building production-ready multi-agent systems.

## Topics Covered

1. Custom orchestration patterns
2. Error handling and retry mechanisms
3. Performance optimization
4. Cost tracking and monitoring
5. Integration with external systems
6. Testing multi-agent systems

In [ ]:
# Setup
from agentmind import Agent, AgentMind
from agentmind.llm import OllamaProvider
from agentmind.tools import Tool
from agentmind.utils.retry import RetryConfig, retry_with_backoff
from agentmind.utils.observability import Tracer, CostTracker
import asyncio
import json
from typing import Dict, List, Any

import nest_asyncio
nest_asyncio.apply()

## Part 1: Custom Orchestration Patterns

Create custom collaboration patterns beyond the default round-robin.

In [ ]:
from agentmind.orchestration import Orchestrator
from agentmind.core.types import Message

class HierarchicalOrchestrator(Orchestrator):
    """Manager agent delegates to worker agents"""
    
    def __init__(self, manager_agent: Agent, worker_agents: List[Agent]):
        self.manager = manager_agent
        self.workers = worker_agents
    
    async def orchestrate(self, task: str, max_rounds: int = 3) -> str:
        messages = []
        
        # Manager analyzes task and creates subtasks
        manager_prompt = f"""Analyze this task and break it into {len(self.workers)} subtasks:
        
        Task: {task}
        
        Create one subtask for each worker. Format as JSON list."""
        
        manager_response = await self.manager.generate(manager_prompt, messages)
        messages.append(Message(role="assistant", content=manager_response, agent=self.manager.name))
        
        # Workers execute subtasks in parallel
        worker_tasks = []
        for i, worker in enumerate(self.workers):
            worker_prompt = f"""Complete this subtask:
            
            Original task: {task}
            Manager's breakdown: {manager_response}
            Your subtask: Subtask {i+1}
            """
            worker_tasks.append(worker.generate(worker_prompt, messages))
        
        worker_results = await asyncio.gather(*worker_tasks)
        
        for worker, result in zip(self.workers, worker_results):
            messages.append(Message(role="assistant", content=result, agent=worker.name))
        
        # Manager synthesizes results
        synthesis_prompt = f"""Synthesize the worker results into a final answer:
        
        Original task: {task}
        Worker results: {json.dumps(worker_results)}
        
        Provide a comprehensive final answer."""
        
        final_result = await self.manager.generate(synthesis_prompt, messages)
        
        return final_result

# Test hierarchical orchestration
async def test_hierarchical():
    llm = OllamaProvider(model="llama3.2")
    
    manager = Agent(
        name="Manager",
        role="manager",
        system_prompt="You are a project manager who delegates tasks and synthesizes results.",
        llm_provider=llm
    )
    
    workers = [
        Agent(name="Worker1", role="analyst", system_prompt="You analyze data.", llm_provider=llm),
        Agent(name="Worker2", role="researcher", system_prompt="You research information.", llm_provider=llm),
        Agent(name="Worker3", role="writer", system_prompt="You write content.", llm_provider=llm),
    ]
    
    orchestrator = HierarchicalOrchestrator(manager, workers)
    result = await orchestrator.orchestrate("Create a market analysis report for electric vehicles")
    
    print("Hierarchical Orchestration Result:")
    print(result)

await test_hierarchical()

## Part 2: Error Handling and Retry Mechanisms

Build robust systems that handle failures gracefully.

In [ ]:
class UnreliableTool(Tool):
    """A tool that sometimes fails (for demonstration)"""
    
    def __init__(self):
        self.attempt = 0
        super().__init__(
            name="unreliable_api",
            description="An API that sometimes fails",
            parameters={"query": {"type": "string"}}
        )
    
    async def execute(self, query: str) -> str:
        self.attempt += 1
        
        # Fail first 2 attempts, succeed on 3rd
        if self.attempt < 3:
            raise Exception(f"API Error: Connection timeout (attempt {self.attempt})")
        
        return json.dumps({"query": query, "result": "Success!", "attempts": self.attempt})

async def test_retry_mechanism():
    """Test retry with exponential backoff"""
    
    tool = UnreliableTool()
    
    # Configure retry
    config = RetryConfig(
        max_retries=5,
        initial_delay=0.5,
        backoff_factor=2.0,
        max_delay=10.0
    )
    
    print("Testing retry mechanism...\n")
    
    try:
        result = await retry_with_backoff(
            lambda: tool.execute(query="test"),
            config
        )
        print(f"Success after retries: {result}")
    except Exception as e:
        print(f"Failed after all retries: {e}")

await test_retry_mechanism()

## Part 3: Performance Optimization

Optimize multi-agent systems for speed and efficiency.

In [ ]:
import time
from functools import lru_cache

# Technique 1: Caching expensive operations
@lru_cache(maxsize=100)
def expensive_computation(input_data: str) -> str:
    """Simulate expensive computation"""
    time.sleep(0.1)  # Simulate delay
    return f"Processed: {input_data}"

# Technique 2: Parallel execution
async def parallel_agent_execution():
    """Execute multiple agents in parallel"""
    llm = OllamaProvider(model="llama3.2")
    
    agents = [
        Agent(name=f"Agent{i}", role="worker", system_prompt="You are a worker.", llm_provider=llm)
        for i in range(3)
    ]
    
    # Sequential execution (slow)
    start = time.time()
    sequential_results = []
    for agent in agents:
        result = await agent.generate("Quick task", [])
        sequential_results.append(result)
    sequential_time = time.time() - start
    
    # Parallel execution (fast)
    start = time.time()
    parallel_tasks = [agent.generate("Quick task", []) for agent in agents]
    parallel_results = await asyncio.gather(*parallel_tasks)
    parallel_time = time.time() - start
    
    print(f"Sequential execution: {sequential_time:.2f}s")
    print(f"Parallel execution: {parallel_time:.2f}s")
    print(f"Speedup: {sequential_time/parallel_time:.2f}x")

await parallel_agent_execution()

## Part 4: Cost Tracking and Monitoring

Track token usage and costs in production.

In [ ]:
async def track_costs():
    """Track costs and performance metrics"""
    
    # Initialize tracker
    tracker = CostTracker()
    tracer = Tracer(session_id="demo-session")
    
    tracer.start()
    tracker.start()
    
    # Run some agent operations
    llm = OllamaProvider(model="llama3.2")
    mind = AgentMind(llm_provider=llm)
    
    agent = Agent(
        name="CostDemo",
        role="assistant",
        system_prompt="You are a helpful assistant."
    )
    mind.add_agent(agent)
    
    # Execute multiple tasks
    tasks = [
        "Explain quantum computing in one sentence.",
        "What is machine learning?",
        "Define artificial intelligence."
    ]
    
    for task in tasks:
        await mind.collaborate(task, max_rounds=1)
    
    tracer.end()
    tracker.end()
    
    # Display metrics
    print("\nPerformance Metrics:")
    print("="*60)
    print(tracer.get_summary())
    
    print("\nCost Metrics:")
    print("="*60)
    print(f"Total tokens: {tracker.total_tokens}")
    print(f"Total cost: ${tracker.total_cost:.4f}")
    print(f"Average cost per request: ${tracker.average_cost:.4f}")

await track_costs()

## Part 5: Integration with External Systems

Connect agents to databases, APIs, and other services.

In [ ]:
import aiohttp

class APITool(Tool):
    """Tool for making HTTP API calls"""
    
    def __init__(self):
        super().__init__(
            name="api_call",
            description="Make HTTP GET request to an API",
            parameters={
                "url": {"type": "string", "description": "API endpoint URL"}
            }
        )
    
    async def execute(self, url: str) -> str:
        """Make async HTTP request"""
        try:
            async with aiohttp.ClientSession() as session:
                async with session.get(url, timeout=10) as response:
                    data = await response.json()
                    return json.dumps(data)
        except Exception as e:
            return json.dumps({"error": str(e)})

class DatabaseTool(Tool):
    """Simulated database tool"""
    
    def __init__(self):
        # Simulated database
        self.db = {
            "users": [
                {"id": 1, "name": "Alice", "email": "alice@example.com"},
                {"id": 2, "name": "Bob", "email": "bob@example.com"},
            ],
            "products": [
                {"id": 1, "name": "Laptop", "price": 999},
                {"id": 2, "name": "Mouse", "price": 29},
            ]
        }
        super().__init__(
            name="query_database",
            description="Query the database",
            parameters={
                "table": {"type": "string", "description": "Table name"},
                "filter": {"type": "object", "description": "Filter criteria"}
            }
        )
    
    async def execute(self, table: str, filter: Dict = None) -> str:
        """Query simulated database"""
        if table not in self.db:
            return json.dumps({"error": f"Table {table} not found"})
        
        results = self.db[table]
        
        # Apply filters
        if filter:
            results = [
                row for row in results
                if all(row.get(k) == v for k, v in filter.items())
            ]
        
        return json.dumps(results)

async def test_integrations():
    """Test external system integrations"""
    llm = OllamaProvider(model="llama3.2")
    
    agent = Agent(
        name="IntegrationAgent",
        role="data_agent",
        system_prompt="You can query databases and APIs to get information.",
        tools=[DatabaseTool()],
        llm_provider=llm
    )
    
    mind = AgentMind(llm_provider=llm)
    mind.add_agent(agent)
    
    result = await mind.collaborate(
        "Find all users in the database and list their names.",
        max_rounds=1
    )
    
    print("Integration Result:")
    print(result)

await test_integrations()

## Part 6: Testing Multi-Agent Systems

Write tests for your agents and tools.

In [ ]:
import pytest

class TestAgent:
    """Test agent behavior"""
    
    @pytest.mark.asyncio
    async def test_agent_creation(self):
        agent = Agent(
            name="TestAgent",
            role="tester",
            system_prompt="You are a test agent."
        )
        assert agent.name == "TestAgent"
        assert agent.role == "tester"
    
    @pytest.mark.asyncio
    async def test_agent_with_tools(self):
        class MockTool(Tool):
            def __init__(self):
                super().__init__(name="mock", description="Mock tool", parameters={})
            
            async def execute(self) -> str:
                return "mock result"
        
        agent = Agent(
            name="ToolAgent",
            role="tester",
            system_prompt="Test",
            tools=[MockTool()]
        )
        
        assert len(agent.tools) == 1
        assert agent.tools[0].name == "mock"

# Mock test execution
print("Example test structure shown above.")
print("Run with: pytest tests/test_agents.py")

## Part 7: Production Deployment Checklist

Before deploying to production:

### 1. Error Handling
- [ ] Implement retry mechanisms
- [ ] Add fallback strategies
- [ ] Log all errors
- [ ] Set up alerting

### 2. Performance
- [ ] Profile token usage
- [ ] Optimize prompts
- [ ] Use caching where appropriate
- [ ] Implement rate limiting

### 3. Monitoring
- [ ] Track costs
- [ ] Monitor response times
- [ ] Log all interactions
- [ ] Set up dashboards

### 4. Security
- [ ] Validate all inputs
- [ ] Sanitize outputs
- [ ] Secure API keys
- [ ] Implement authentication

### 5. Testing
- [ ] Unit tests for tools
- [ ] Integration tests for agents
- [ ] End-to-end tests
- [ ] Load testing

### 6. Documentation
- [ ] API documentation
- [ ] Deployment guide
- [ ] Runbooks
- [ ] Architecture diagrams

## Summary

You've learned:

1. Custom orchestration patterns for complex workflows
2. Robust error handling with retry mechanisms
3. Performance optimization techniques
4. Cost tracking and monitoring
5. Integration with external systems
6. Testing strategies for multi-agent systems

## Next Steps

- Deploy your first production system
- Contribute to AgentMind
- Join the community
- Share your use cases

## Resources

- [Production API Guide](../API.md)
- [Docker Deployment](../DOCKER.md)
- [Performance Guide](../PERFORMANCE.md)
- [Troubleshooting](../TROUBLESHOOTING.md)